In [ ]:
# Lets import the needed libraries. There are some quick annotations for what the unusual ones are used for.

import csv
import pandas as pd
import numpy as np
import re
import os
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification # An NLP model we use for sentiment analysis, it is pretrained. Details are in README.
from huggingface_hub import login # Our login/API key for hugging face, not strictly needed but helps run smoother.
from dotenv import load_dotenv # Adding secret API keys.
from datetime import datetime
import statsmodels.api as sm # For our regression model.
import matplotlib.pyplot as plt
import sqlite3 # For the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.
from sklearn.preprocessing import MinMaxScaler # For data normalisation.


# Below we load our API keys from the .env. You will require your own API keys, this is for the purpose of hiding mine.
# In this script we only use a HuggingFace key, althought this is not strictly required for the code to run, it does make it run smoother.

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)

In [ ]:
# In this cell we quickly load our data. We combine the Guardian News data, Alphavantage data and the NewsAPI data into one dataframe (news) via concat.

News_api_data = pd.read_csv('../Data/Data_Pre_Sentiment/NewsAPI_data.csv',encoding='utf-8') # Has some ecoding issues earlier, so just ensuring it is utf-8.
guardian_data = pd.read_csv('../Data/Data_Pre_Sentiment/guardian_data.csv',encoding='utf-8')
alphavantage_data = pd.read_csv('../Data/Data_Pre_Sentiment/alphavantage_data.csv',encoding='utf-8')
guardian_data = guardian_data.rename(columns={'headline': 'title', 'body_text': 'content'}, inplace=True) # Quickly rename Guardian and alphavantage columns so it will merge properly.
alphavantage_data = alphavantage_data.rename(columns={'summary':'content'})
news = pd.concat([guardian_data,News_api_data,alphavantage_data])



# We also load the financial data as well.

brent = pd.read_csv('../Data/Data_Pre_Sentiment/Brent_data.csv',encoding='utf-8')
wti = pd.read_csv('../Data/Data_Pre_Sentiment/WTI_data.csv',encoding='utf-8')
ovx = pd.read_csv('../Data/Data_Pre_Sentiment/ovx_data.csv',encoding='utf-8')
vix = pd.read_csv('../Data/Data_Pre_Sentiment/vix_data.csv',encoding='utf-8')
sp500 = pd.read_csv('../Data/Data_Pre_Sentiment/S&P500_data.csv',encoding='utf-8')


In [ ]:
# This cell runs our sentiment analysis models over our news data.
# We use the ProsusAI/finbert model.
# Quick to run, only taking 2 minute (on my laptop).

prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert') # Defining the model and tokenizer.
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

# We use batching to speed up the process, if you are on a powerfull computer, you can increase the batch_size to speed up the run time.
nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

# Creating a function to run over the title and content columns, it returns out the sentiment scores.
def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

news['title_score_prosus'] = sentiment(news['title'])
news['content_score_prosus'] = sentiment(news['content'])
news['final_score_prosus'] = news[['title_score_prosus','content_score_prosus']].mean(axis=1) # We create an additional column for an average score (of tile and content).

In [ ]:
# Quickly saving the news and sentiment data so we dont have to rerun the above. 
# We load it again in the next cell.

news.to_csv('../Data/Data_Post_Sentiment/important_one.csv', index=False, encoding='utf-8')

In [ ]:
# Merging later on throws a date data type error, so we again ensure everything is the same date data type here.

news = pd.read_csv('../Data/Data_Post_Sentiment/important_one.csv', encoding='utf-8')
news['date'] = pd.to_datetime(news['date'], format='%Y%m%d')
brent['date'] = pd.to_datetime(brent['date'])
wti['date'] = pd.to_datetime(wti['date'])
ovx['date'] = pd.to_datetime(ovx['date'])
vix['date'] = pd.to_datetime(vix['date'])
sp500['date'] = pd.to_datetime(sp500['date'])

In [ ]:
# We create new dataframes with the average daily sentiment using the final average score, title score alone, and content score alone.
# We can compare the sentiment between titles and content this way.
# We get the mean, but standard deviation and count will also be usefull

sentiment_by_day_for_avg = pd.DataFrame(news.groupby('date')['final_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_title = pd.DataFrame(news.groupby('date')['title_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_content = pd.DataFrame(news.groupby('date')['content_score_prosus'].agg(['mean','std','count']))

sentiment_by_day_for_avg.reset_index(inplace=True) # Resetting the index.
sentiment_by_day_for_title.reset_index(inplace=True)
sentiment_by_day_for_content.reset_index(inplace=True)



In [ ]:
# Graph 1: This is a line graph comparing the sentiment scores if we use titles alone, content alone, or the average of them.

fig, ax1 = plt.subplots(figsize=(20, 5))

ax1.plot(sentiment_by_day_for_avg['date'], sentiment_by_day_for_avg['mean'], color='#384358', linewidth=2.5, label='Total Average Per Day')
ax1.plot(sentiment_by_day_for_title['date'], sentiment_by_day_for_title['mean'], color='#BC1139', linewidth=2.5, label='Headline Score Per Day')
ax1.plot(sentiment_by_day_for_content['date'], sentiment_by_day_for_content['mean'], color='#739ab9', linewidth=2.5, label='Content Score Per Day')

ax1.set_ylabel('Sentiment Average\n(Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
ax1.legend(loc='upper left')
plt.xticks(rotation=45, fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace')
plt.grid(linestyle='--', alpha=0.5)

plt.text(sentiment_by_day_for_avg['date'].iloc[20], -0.70, 'We see sustained negative sentiment, that improves slighty towards current day.'
         , color="#000000",fontsize=9,fontfamily='monospace')

plt.title('Averaged Sentiment VS Title Only VS Content Only', fontsize=13, fontfamily='monospace')
plt.tight_layout()
plt.show()

plt.savefig('../Outputs/Graphs/Averaged_Sentiment _VS_Title_Only_VS_Content_Only.png', dpi=300, bbox_inches='tight')

In [ ]:
# We merge the average sentiment scores with brent,wti,ovx and vix.

pre_complete1 = pd.merge(sentiment_by_day_for_avg, brent ,on='date', how='inner')
pre_complete2 = pd.merge(pre_complete1, wti,on='date', how='inner')
pre_complete3 = pd.merge(pre_complete2, vix,on='date', how='inner')
pre_complete3 = pre_complete3.rename(columns={'mean': 'sentiment_mean','std': 'sentiment_std','daily_avg_x': 'brent','daily_avg_y': 'wti','daily_avg': 'vix'})
pre_complete4 = pd.merge(pre_complete3, ovx,on='date', how='inner')
pre_complete5 = pd.merge(pre_complete4, sp500,on='date', how='inner')
complete = pre_complete5.rename(columns={'daily_avg_x':'ovx','daily_avg_y':'sp500'})



In [ ]:
# While this section could be done in python, for the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.

conn = sqlite3.connect('newsdata.db')
complete.to_sql('news_and_data',conn,if_exists='replace',index=False)
conn.commit()

# We calculate a moving average to make underlying trends easier to spot.
def movingaverage(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       AVG({column}) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
                       AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()
    
movingaverage('sentiment_mean','sentiment_mean_moving_average')
movingaverage('ovx','ovx_moving_average')
movingaverage('vix','vix_moving_average')
movingaverage('wti','wti_moving_average')
movingaverage('brent','brent_moving_average')

# We invert some columns so that scales can be presenting the same way.
def invert(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       -{column} AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()

invert('sentiment_mean_moving_average','sentiment_mean_moving_average_inversed')

# We lag some columns to see if previous day x predicts y
def lag(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (LAG({column}) OVER (ORDER BY date)) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

lag('brent','lagged_brent')
lag('wti','lagged_wti')
lag('vix','lagged_vix')
lag('ovx','lagged_ovx')
lag('sentiment_mean','lagged_sentiment_mean')
lag('sp500','lagged_sp500')

# We take the absolute values.
def abs_value(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        ABS({column}) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

abs_value('sentiment_mean','sentiment_mean_abs')

# We calculate percentage change.
def percentage_change(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (((({column}) - (LAG({column}) OVER(ORDER BY date))) / (LAG({column}) OVER(ORDER BY date)))*100) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()
     
percentage_change('brent','brent_percent')
percentage_change('wti','wti_percent')
percentage_change('vix','vix_percent')
percentage_change('ovx','ovx_percent')
percentage_change('sp500','sp500_percent')

In [ ]:
# We take everythign in the SQL table and make it into a pandas dataframe.

All_needed_data = pd.read_sql('''SELECT * FROM news_and_data''',conn)
All_needed_data['date'] = pd.to_datetime(All_needed_data['date'], format='ISO8601')

In [ ]:
# Regression 1: Basic exploratory regression
# checking is sentiment is persistant, can oil yesterday move sentiment, and does braoder market fear drive sentiement.



temp = All_needed_data.copy()
temp.dropna(inplace=True) # Wont run if null values.

x1 = temp[['lagged_wti','lagged_vix','lagged_sentiment_mean']] # Our independent variables  If we use lagged_ovx, we introduce alot of multicolinearity as wti (oil) and ovx (oil volatility)
y1 = temp['sentiment_mean'] # Our dependent variable

x1 = sm.add_constant(x1)
model1 = sm.OLS(y1, x1).fit()
print(model1.summary())

# We expalin 14-20% of the varience in sentiment mean (R^2 vs adj R^2), with our model being jointly significant at the 5% level.
# Both yesterdays wti and vix value are statistically insignificant, thus sentiment does not follow oil and market fear movements day after day.
# Thus news is not just reacting to yesterdays barrel price of oil.

In [ ]:
# Regression 2: Adding dummies and interactive terms.
# 40% explainability with 3 statistically significant variables.

temp = All_needed_data.copy()
temp.dropna(inplace=True) # Wont run if null values.

temp['disagreement'] = temp['sentiment_std'] * temp['count'] # Disagreement around sentiment
temp['momentum'] = temp['sentiment_mean'] - temp['lagged_sentiment_mean'] # Direction

temp['high_vix_dummy'] = np.where(temp['vix'] > 25, 1, 0) # Creating a dummy for higher market fear/volatility.


x2 = temp[['lagged_wti','high_vix_dummy','disagreement','momentum']]
y2 = temp['sentiment_mean'] # Our dependent variable


x2 = sm.add_constant(x2)
model2 = sm.OLS(y2, x2).fit(cov_type='HC3') #Using HC3 as it is better for smaller samples.
print(model2.summary())


# This model doubles our explainability, at around 50. Overally the model is jointly significant at the 5% level.
# Our dummy is significant. Days with higher broader market fear give a lower sentimetn by 0.01 points lower.
# The disagreement term is significant.
# The momentum term is significant. If sentiment improved yesterday it tends to keep improving today.


In [ ]:
# Graph 2: Overview of oil and sentiment.

fig, ax1 = plt.subplots(figsize=(20,6))

ax1.bar(All_needed_data['date'], All_needed_data['sentiment_mean'], color='#2D244D', width=0.8, label='Daily Sentiment')
ax1.axhline(0, color='black')
ax1.set_ylabel('Sentiment by day\n(Lower is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
ax1.set_ylim(-1, 1)

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['brent'], color='#FF5B04', linewidth=2.5, label='Brent Crude (USD)')
ax2.plot(All_needed_data['date'], All_needed_data['wti'], color='#C2441C', linewidth=2.5, label='WTI Crude (USD)')
ax2.set_ylabel('Oil Price (USD/barrel)', fontsize=11, fontfamily='monospace')
plt.xticks(fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace',fontsize=7)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.text(sentiment_by_day_for_avg['date'].iloc[13], 70, 'Oil surged ~55% from Mar 1 to Mar 22 as the conflict escalated.\nWe see sustained negative sentiment throughout.\nMarkets priced in fear before some optimism.'
         , color="#000000",fontsize=9,fontfamily='monospace')

plt.title('Overveiw of News Sentiment and Crude Oil Price Over the Iran War', fontsize=13, fontfamily='monospace')
plt.show()

plt.savefig('../Outputs/Graphs/Overveiw_of_News_Sentiment_and_Crude_Oil_Price_Over_the_Iran_War.png', dpi=300, bbox_inches='tight')

In [ ]:
# Graph 3: Comparing oil volatility vs more general market fear.

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(All_needed_data['date'], All_needed_data['ovx_moving_average'], color='#BC1139', linewidth=2.5, label='OVX moving average (Oil Volatility)')
ax.plot(All_needed_data['date'], All_needed_data['vix_moving_average'], color='#739ab9', linewidth=2.5 , label='VIX moving average (Borader Market)')

ax.set_ylabel('Volatility Index',fontfamily='monospace',fontsize=11)
ax.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
plt.xticks(fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace',fontsize=7)


plt.text(All_needed_data['date'].iloc[35], 45, 'OVX peaked near 120 (over 4x the VIX).\nInvestors feared an oil shock specifically,\nnot a broader market crash')

plt.grid(linestyle='--', alpha=0.5)
ax.set_title('Oil Fear (OVX) vs Broad Market Fear (VIX) During Iran War\n''(The war was an energy shock, not a systemic market panic)',fontsize=13
             ,fontfamily='monospace')
ax.legend()
plt.tight_layout()

plt.savefig('../Outputs/Graphs/OVX_vs_VIX.png', dpi=300, bbox_inches='tight')

In [ ]:
# Graph 4: We compare the OVX index against our measure of fear.

fig, ax = plt.subplots(figsize=(14, 6))

minmax = All_needed_data[['sentiment_mean_moving_average_inversed','ovx_moving_average']]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(minmax.to_numpy())
scaled_data = pd.DataFrame(scaled_data, columns=['scaled_sentiment_mean_moving_average_inversed','scaled_ovx_moving_average'])

ax.plot(All_needed_data['date'], scaled_data['scaled_sentiment_mean_moving_average_inversed'], color='#BC1139', linewidth=2.5, label='News Sentiment Inversed as fear (MA)')
ax.set_ylabel('Fear In Normalised Arbitrary Units\n(Higher More Volatile)', fontsize=11, fontfamily='monospace')
ax.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')
plt.xticks(fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace',fontsize=7)

ax.plot(All_needed_data['date'], scaled_data['scaled_ovx_moving_average'], color='#739ab9', linewidth=2.5, label='OVX (MA)')

ax.legend()

plt.text(All_needed_data['date'].iloc[20], 0.25, 'News sentiment signalled fear before OVX did.\nAs OVX caught up, sentiment partially recovered.\nMarkets were slower to price in what headlines implied.'
         , color="#000000",fontsize=11,fontfamily='monospace')

plt.title('OVX Fear Index vs Fear From News', fontsize=13, fontfamily='monospace')
plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

plt.savefig('../Outputs/Graphs/OVX_vs_news_sentiment.png', dpi=300, bbox_inches='tight')

In [ ]:
# Graph 5: Exploring sentiment vs lagged WTI oil.

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(All_needed_data['date'], All_needed_data['sentiment_mean_moving_average_inversed'], color='#BC1139', linewidth=2.5, label='News Sentiment Per Day')
ax1.set_ylabel('Sentiment Per Day\n(Higher is Worse)', fontsize=11, fontfamily='monospace')
ax1.set_xlabel('Date (start to present date)', fontsize=11, fontfamily='monospace')

ax2 = ax1.twinx()
ax2.plot(All_needed_data['date'], All_needed_data['wti_moving_average'], color='#739ab9', linewidth=2.5, label='Lagged WTI Oil USD')
ax2.set_ylabel('Brent USD\n(Higher is Worse)', fontsize=11, fontfamily='monospace')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.xticks(fontsize=7, fontfamily='monospace')
plt.yticks(fontfamily='monospace',fontsize=7)

plt.text(All_needed_data['date'].iloc[18], 80, 'Oil prices trail sentiment.\nWhen sentiment rebounds in mid april, oil follows.\nThis supports the view that news leads price, not the reverse.'
         , color="#000000",fontsize=11,fontfamily='monospace')


plt.title('Sentiment Per Day vs WTI Oil', fontsize=13, fontfamily='monospace')
plt.tight_layout()
plt.show()

plt.savefig('../Outputs/Graphs/Sentiment_vs_wti.png', dpi=300, bbox_inches='tight')